In [1]:
#import relevant libraries
import os
#from scipy import stats

import numpy as np
#import scipy as sp
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.cm
import seaborn as sns
import dabest
import NLCLIMB
import NLMATH_multistate as NLMATH  # Using the new multistate version
import itertools
from datetime import datetime
date = datetime.today().strftime('%Y%m%d')
from statistics import mean
from textwrap import wrap


import dabest
import plotly.express as px 
from plotly.subplots import make_subplots
import plotly.graph_objects as go
from plotly.graph_objects import Layout
#from mpl_toolkits.axes_grid1.inset_locator import inset_axes

#NOTE: SUPPRESSES WARNINGS!

import warnings


warnings.simplefilter(action="ignore", category=RuntimeWarning)
warnings.simplefilter(action="ignore", category=UserWarning)
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)
warnings.simplefilter(action='ignore', category=FutureWarning)

Pre-compiling numba functions for DABEST...


Compiling numba functions: 100%|██████████| 11/11 [00:00<00:00, 51.05it/s]

Numba compilation complete!


In [2]:
#initial file processing
workcomp = "C:\\Users\\User"
computer2 = "C:\\Users\\lnico"
officecomp = "C:\\Users\\Star"
homecomp = "D:"
titledpath = workcomp

#Comment depending on project type
#eopn3work = "N"
eopn3work = "Y"
eopn3work = "tgt"

if eopn3work == "Y":
    whomst ="NL"
    filedir = "\\ACC Lab Dropbox\\ACC Lab\\Nicole Lee\\eOPN3 manuscript\\Data compilation\\2. Processed\\" + whomst + "\\"
    savedir = titledpath + "\\ACC Lab Dropbox\\ACC Lab\\Nicole Lee\\eOPN3 manuscript\\Data compilation\\3. Compiled\\"  + whomst + "\\"
    
if eopn3work == "tgt":
    filedir = "\\ACC Lab Dropbox\\ACC Lab\\Nicole Lee\\eOPN3 manuscript\\Data compilation\\Together\\2. Processed\\" 
    savedir = titledpath + "\\ACC Lab Dropbox\\ACC Lab\\Nicole Lee\\eOPN3 manuscript\\Data compilation\\Together\\3. Compiled\\" 
    
else:
    filedir = "\\ACC Lab Dropbox\\ACC Lab\\Nicole Lee\\Data Compilation\\Falling_New\\"
    savedir = titledpath + filedir + "Compilation with delta\\2025meandiffcollection\\"
    
openPath = titledpath + filedir
files = os.listdir(openPath)

#identifying genotypes
responder = "eOPN3"
respondercsv = responder + ".csv"
wt = "w1118"

In [3]:
lstnew=[]

#lstnew should be the list of names you want to process the files with. Only choose one

#if you want to process all the names in the filedir

# for file_no in os.listdir(openPath): 
#     if respondercsv in file_no and "w1118" not in file_no :   
#         f = os.path.join(openPath, file_no)
#         dfe=pd.read_csv(f)
#         exptdf = dfe.drop(dfe.columns[[0]],axis = 1)
#         driver = file_no.split(" ")[0]
#         lstnew.append(driver)
# lst = lstnew.copy()

#processing ONLY specific names
lst = ["elav", "nSyb", "OK371", "Piezo", "vGAT", "Cha-6793"]

print(lst)

['elav', 'nSyb', 'OK371', 'Piezo', 'vGAT', 'Cha-6793']


In [ ]:
# Process data with all three state comparisons
all_comparisons = pd.DataFrame()
comparison_types = ['DARK-FULL', 'DARK-RECOVERY']

for n in lst:
    driver = n
    print(f"Processing {n}...")
    transgenic = driver + " x " + responder
    filename = openPath + transgenic + ".csv"
    filenamewt = openPath + wt+"_"+ transgenic + ".csv"

    dfe=pd.read_csv(filename)
    dfw= pd.read_csv(filenamewt)

    exptdf = dfe.drop(dfe.columns[[0]],axis = 1)
    wtdf = dfw.drop(dfw.columns[[0]],axis = 1)

    dfexpt = NLCLIMB.fivesecondrule(NLCLIMB.generation(exptdf, driver))
    dfwt = NLCLIMB.fivesecondrule(NLCLIMB.generation(wtdf, wt))
    
    # Calculate metrics for all states (including Recovery)
    df_f = NLMATH.fallingocc(dfexpt, dfwt).reset_index(drop=True) 
    df_sp = NLMATH.ospeed(dfwt, dfexpt).reset_index(drop=True)
    df_bsp = NLMATH.bspeed(NLMATH.boutspeed(dfexpt), NLMATH.boutspeed(dfwt)).reset_index(drop=True)
    df_h = NLMATH.totalheight(dfexpt, dfwt).reset_index(drop=True)
    df_maxv = pd.concat([NLMATH.maxvelocity(dfexpt, "Expt"), NLMATH.maxvelocity(dfwt, "WT")], axis = 0).reset_index(drop=False)
    
    # Process each comparison type
    for comparison in comparison_types:
        print(f"  Computing {comparison} comparison...")
        
        # Use the new multistate functions
        dff2_prop = NLMATH.deltaversion_binary_multistate(df_f, "binary_fallvalue", "fallprop", comparison)
        dff2_number = NLMATH.deltaversion_multistate(df_f, "Fall", "fallnumber", comparison)
        dfs2 = NLMATH.deltaversion_multistate(df_sp, "Velocity", "speed", comparison)
        dfh2 = NLMATH.deltaversion_multistate(df_h, "Y", "height", comparison)
        dfbs2 = NLMATH.deltaversion_multistate(df_bsp, "BSpeed", "bspeed", comparison)
        dfmv2 = NLMATH.deltaversion_multistate(df_maxv, "maxvelocity", "maxvelocity", comparison)
        
        # Combine all metrics for this comparison
        dftotal = pd.concat([dff2_prop, dff2_number, dfs2, dfh2, dfbs2, dfmv2], axis = 1)
        dftotal['MBON'] = n
        dftotal['State_Comparison'] = comparison
        dftotal['comparison_type'] = comparison  # Keep for consistency
               
        # Save individual comparison file
        dftotal.set_index("MBON", inplace = True)
        comparison_suffix = comparison.replace('-', '_')
        dftotal.to_csv(savedir + n + " x " + responder + f" allstats_{comparison_suffix}.csv")
        dftotal.reset_index(inplace=True)
    
print("\nProcessing complete!")

In [37]:

comparison_type = 'DARK-RECOVERY'

# Filter data based on comparison type
if comparison_type == 'DARK-FULL':
    df6 = df_bsp[(df_bsp['ExperimentState'] != "Recovery")]
    x1_level = ["Dark", "Full"]

elif comparison_type == 'DARK-RECOVERY':
    df6 = df_bsp[(df_bsp['ExperimentState'] != "Full")]
    x1_level = ["Dark", "Recovery"]
else:
    raise ValueError("comparison_type must be 'DARK-FULL', 'FULL-RECOVERY', or 'DARK-RECOVERY'")

name = []
if any(df6["BSpeed"].isnull()):
    name = df6[df6["BSpeed"].isnull()]['index'].tolist()
    
dfsp_db = df6[~df6['index'].isin(name)]
        
dfsp_db2 = dabest.load(data = dfsp_db, x = ["ExperimentState", "Type"], y = "BSpeed",  delta2 = True, experiment = "Type",
                        experiment_label = ['WT', 'Expt'], x1_level = x1_level, paired = "baseline", id_col="index" )
dfstatstest = dfsp_db2.hedges_g.statistical_tests
    
if dfstatstest['control'][0].split(" ")[1] == "WT" and dfstatstest['control'][1].split(" ")[1] == "Expt":
    dfdiff = pd.DataFrame({
        "bspeed" + "_bootstrap": dfsp_db2.hedges_g.delta_delta.bootstraps_delta_delta.tolist(), 
        "bspeed" + "_deltag": round(dfsp_db2.hedges_g.delta_delta.difference,3),
        "comparison_type": comparison_type
    })
dfstatstest

ZeroDivisionError: division by zero

In [40]:
NLMATH.boutspeed(dfexpt)

,Seconds,ExperimentState,vGAT BSpeed_1,vGAT BSpeed_2,vGAT BSpeed_3,vGAT BSpeed_4,vGAT BSpeed_5,vGAT BSpeed_6,vGAT BSpeed_7,vGAT BSpeed_8,...,vGAT BSpeed_156,vGAT BSpeed_158,vGAT BSpeed_159,vGAT BSpeed_160,vGAT BSpeed_161,vGAT BSpeed_162,vGAT BSpeed_163,vGAT BSpeed_164,vGAT BSpeed_165,vGAT BSpeed_166
0,3.0,Dark,4.962484,9.869863,7.331783,9.708376,16.511507,6.070504,12.623916,14.549168,...,8.418781,7.560679,3.161800,2.684607,NaN,7.325227,7.949531,9.864907,11.652914,4.513808
1,3.2,Dark,11.034325,14.365676,13.527447,9.236726,4.342041,5.794187,9.843836,13.639923,...,4.677483,12.552358,4.975765,2.256839,2.504216,3.024658,10.742222,5.923085,18.926546,NaN
2,3.4,Dark,5.713716,12.221649,14.761403,6.949615,10.201161,5.844934,8.328541,12.024092,...,2.809387,7.201837,NaN,NaN,2.864904,NaN,9.048727,7.386609,20.341218,NaN
3,3.6,Dark,3.655102,NaN,13.405245,4.752877,12.521231,6.188277,7.994248,12.777744,...,4.639365,13.705117,NaN,NaN,2.270416,NaN,6.195214,6.806225,21.307821,NaN
4,3.8,Dark,11.175522,12.584158,11.883914,7.842196,13.688415,5.851671,8.010648,13.061652,...,5.745984,6.248275,6.575129,3.014367,2.557269,NaN,6.864401,4.666012,20.626946,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
287,65.0,Recovery,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
288,65.2,Recovery,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
289,65.4,Recovery,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
290,65.6,Recovery,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [34]:
dfsp_db[dfsp_db['Type'] == "Expt"]

,index,BSpeed,ExperimentState,Type,genre
10,vGAT BSpeed_11,7.149741,Dark,Expt,Dark Expt
23,vGAT BSpeed_25,6.121834,Dark,Expt,Dark Expt
228,vGAT BSpeed_11,3.815548,Recovery,Expt,Recovery Expt
241,vGAT BSpeed_25,10.820771,Recovery,Expt,Recovery Expt


In [39]:
df6[df6['Type'] == "Expt"]

,index,BSpeed,ExperimentState,Type,genre
0,vGAT BSpeed_1,8.568225,Dark,Expt,Dark Expt
1,vGAT BSpeed_2,10.834557,Dark,Expt,Dark Expt
2,vGAT BSpeed_3,8.527566,Dark,Expt,Dark Expt
3,vGAT BSpeed_4,6.245330,Dark,Expt,Dark Expt
4,vGAT BSpeed_5,7.912270,Dark,Expt,Dark Expt
...,...,...,...,...,...
322,vGAT BSpeed_162,NaN,Recovery,Expt,Recovery Expt
323,vGAT BSpeed_163,NaN,Recovery,Expt,Recovery Expt
324,vGAT BSpeed_164,NaN,Recovery,Expt,Recovery Expt
325,vGAT BSpeed_165,NaN,Recovery,Expt,Recovery Expt


In [14]:
NLMATH.deltaversion_binary_multistate(df_bsp, "BSpeed", "bspeed", comparison)

,bspeed_bootstrap,bspeed_deltag,comparison_type
0,1.743546,1.645,DARK-RECOVERY
1,2.263263,1.645,DARK-RECOVERY
2,-1.506320,1.645,DARK-RECOVERY
3,1.616398,1.645,DARK-RECOVERY
4,2.714607,1.645,DARK-RECOVERY
...,...,...,...
4995,6.560565,1.645,DARK-RECOVERY
4996,-2.712415,1.645,DARK-RECOVERY
4997,-0.840618,1.645,DARK-RECOVERY
4998,5.831150,1.645,DARK-RECOVERY


In [ ]:
df_bsp = NLMATH.bspeed(NLMATH.boutspeed(dfexpt), NLMATH.boutspeed(dfwt)).reset_index(drop=True)